# Video Object Indexer: Kaggle run

Runs the whole pipeline from a clean machine: install, download the FineVideo files, index every video, write `results.jsonl.gz`.
Use it two ways:
* **Quick test** (interactive "Run All"): leave `MAX_VIDEOS = 6` and `DATA_PATTERN` on one file. A few minutes.
* **Final run** (**Save Version -> Save & Run All**): set `MAX_VIDEOS = 0` and the 50-file `DATA_PATTERN`. Runs headless, up to 12 hours.

**Before you press run**
1. Right panel, Session options: **Accelerator = GPU T4 x2** (not P100) and **Internet = On**.
2. Add-ons -> Secrets: `HF_TOKEN` must exist **and be switched on for this notebook**.
3. Your GitHub repo must be **public**. Set `GITHUB_USER` in the next cell.
4. Final run only: you have already checked `SAMPLES_PER_SEC` against the slow every-frame reference (set `RUN_ACCURACY_CHECK = True` once to do that).

Outputs (Output tab after the run): `results.jsonl.gz`, `hardware_info.txt`, `run_log.txt`, `run_summary.json`.

In [1]:
# ----------------------------- CONFIG: edit this cell only -----------------------------
GITHUB_USER = "urwa-khalid"            # <-- CHANGE: your GitHub username
REPO_NAME   = "video-object-indexer"
BRANCH      = "main"

# which dataset files to download
DATA_PATTERN = "data/train-0000[0-1]-*"      # quick test: one file.  Final run: "data/train-000[0-4]*"  (the first 50 files)
MAX_VIDEOS   = 6                         # 0 = every video.  Quick test: 6.  Final run: 0

# run.py settings. None = use the default written in the repo
SAMPLES_PER_SEC = None                   # frames per second sent to the model (set it after your accuracy check)
WORKERS         = None                   # e.g. 2, 3, 4 on Kaggle's 4 cores
DECODE_THREADS  = None                   # e.g. 1 or 2
BATCH           = None                   # e.g. 16, 32
EXTRA_ARGS      = ""                     # e.g. "--fp32"

# optional accuracy check against the slow every-frame reference (takes a while; see the section below)
RUN_ACCURACY_CHECK = True
ACC_FILES    = ["train-00000-of-01357.parquet", "train-00001-of-01357.parquet"]
ACC_VIDEOS   = 12                           # about 50 minutes of video per file; more seconds = more trustworthy scores
ACC_K_VALUES = [2, 3, 4, 6]                 # frames per second to compare

# housekeeping
REPO_URL        = None                   # None -> https://github.com/<GITHUB_USER>/<REPO_NAME>.git
DATA_ROOT       = "/tmp/finevideo"       # scratch disk: wiped when the session ends
SKIP_DOWNLOAD   = False                  # True if the files are already in DATA_ROOT/data
FORCE_REINSTALL = False
ALLOW_CPU       = False                  # keep False: refuse to start a long run with no GPU

In [2]:
import glob, gzip, json, os, shutil, subprocess, sys, time

OUT      = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("out")
REPO_DIR = "/tmp/repo"
VENV     = "/tmp/venv"
PY       = f"{VENV}/bin/python"
DATA_DIR = f"{DATA_ROOT}/data"
REPO_URL = REPO_URL or f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
os.makedirs(OUT, exist_ok=True)
LOG = os.path.join(OUT, "run_log.txt")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

def sh(cmd, check=True):
    """Run a shell command, show its output live, copy it to run_log.txt, stop the notebook if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    with open(LOG, "a") as f:
        f.write(f"$ {cmd}\n")
        for line in p.stdout:
            print(line, end="", flush=True)
            f.write(line)
    rc = p.wait()
    if rc != 0 and check:
        raise RuntimeError(f"command failed (exit {rc}): {cmd}")
    return rc

perf = ""
if WORKERS:         perf += f" --workers {WORKERS}"
if DECODE_THREADS:  perf += f" --decode-threads {DECODE_THREADS}"
if BATCH:           perf += f" --batch {BATCH}"
extra = perf + (f" --samples-per-sec {SAMPLES_PER_SEC}" if SAMPLES_PER_SEC is not None else "") + f" {EXTRA_ARGS}"
print("output folder:", OUT)

output folder: /kaggle/working


In [3]:
# 1. what machine is this? (copy these lines into HARDWARE.md later)
cmds = ["nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv",
        "lscpu | grep -E 'Model name|^CPU\\(s\\)'", "nproc", "free -g | head -2",
        "df -h /tmp /kaggle/working 2>/dev/null || df -h /tmp", "lsblk -d -o NAME,ROTA 2>/dev/null",
        "lsb_release -d 2>/dev/null", "python3 --version"]
report = ""
for c in cmds:
    r = subprocess.run(c, shell=True, capture_output=True, text=True, executable="/bin/bash")
    report += f"$ {c}\n{r.stdout}{r.stderr}\n"
open(os.path.join(OUT, "hardware_info.txt"), "w").write(report)
print(report)
if "P100" in report:
    print("WARNING: this is a P100. Recent PyTorch builds may not support it. Choose GPU T4 x2.")

$ nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04

$ lscpu | grep -E 'Model name|^CPU\(s\)'
CPU(s):                                  4
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz

$ nproc
4

$ free -g | head -2
               total        used        free      shared  buff/cache   available
Mem:              31           1          25           0           5          30

$ df -h /tmp /kaggle/working 2>/dev/null || df -h /tmp
Filesystem      Size  Used Avail Use% Mounted on
overlay         8.0T  5.6T  2.5T  70% /
/dev/loop2       20G  136K   20G   1% /kaggle/working

$ lsblk -d -o NAME,ROTA 2>/dev/null
NAME  ROTA
loop0    0
loop1    0
loop2    0
sda      0
sdb      0

$ lsb_release -d 2>/dev/null
Description:	Ubuntu 24.04.4 LTS

$ python3 --version
Python 3.13.16




In [4]:
# 2. get the code
if os.path.isdir(f"{REPO_DIR}/.git"):
    sh(f"git -C {REPO_DIR} pull --ff-only")
else:
    sh(f"rm -rf {REPO_DIR} && git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
sh(f"git -C {REPO_DIR} log -1 --format='code version: %h %s'")

$ rm -rf /tmp/repo && git clone --depth 1 --branch main https://github.com/urwa-khalid/video-object-indexer.git /tmp/repo
Cloning into '/tmp/repo'...
$ git -C /tmp/repo log -1 --format='code version: %h %s'
code version: a481786 Add video object indexer


0

In [5]:
# 3. Python 3.11 + the exact packages from requirements.txt (same as the grader). uv fetches Python 3.11 if Kaggle has another version.
def env_ready():
    return os.path.exists(PY) and subprocess.run(
        [PY, "-c", "import ultralytics, av, torch, cv2, pyarrow"], capture_output=True).returncode == 0

if FORCE_REINSTALL or not env_ready():
    sh("(apt-get update -qq && apt-get install -y -qq libgl1 libglib2.0-0) > /dev/null 2>&1 || echo 'apt-get skipped'", check=False)
    sh(f"{sys.executable} -m pip install -q uv")
    sh(f"rm -rf {VENV} && {sys.executable} -m uv venv {VENV} --python 3.11")
    sh(f"{sys.executable} -m uv pip install -q -p {PY} -r {REPO_DIR}/requirements.txt huggingface_hub")
sh(f"{PY} -c \"import sys, torch; print('python', sys.version.split()[0], '| torch', torch.__version__, "
   f"'| cuda', torch.cuda.is_available(), '| gpus', torch.cuda.device_count())\"")

$ (apt-get update -qq && apt-get install -y -qq libgl1 libglib2.0-0) > /dev/null 2>&1 || echo 'apt-get skipped'
$ /usr/bin/python3 -m pip install -q uv
$ rm -rf /tmp/venv && /usr/bin/python3 -m uv venv /tmp/venv --python 3.11
 Downloaded cpython-3.11.18-linux-x86_64-gnu (download)
Using CPython 3.11.18
Creating virtual environment at: /tmp/venv
Activate with: source /tmp/venv/bin/activate
$ /usr/bin/python3 -m uv pip install -q -p /tmp/venv/bin/python -r /tmp/repo/requirements.txt huggingface_hub
$ /tmp/venv/bin/python -c "import sys, torch; print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| gpus', torch.cuda.device_count())"
python 3.11.18 | torch 2.8.0+cu128 | cuda True | gpus 2


0

In [6]:
# 4. the official yolo11l.pt weights + a quick test of every GPU
sh(f"cd {REPO_DIR} && {PY} download_weights.py")
n_gpu = int(subprocess.run([PY, "-c", "import torch; print(torch.cuda.device_count())"],
                           capture_output=True, text=True).stdout.strip() or 0)
if n_gpu == 0 and not ALLOW_CPU:
    raise RuntimeError("PyTorch sees no GPU. Either the accelerator is off (Session options -> Accelerator -> GPU T4 x2) "
                       "or the driver is too old for this PyTorch build (see the torch / cuda line above). Stop here and tell your helper.")
print("GPUs visible:", n_gpu)

$ cd /tmp/repo && /tmp/venv/bin/python download_weights.py
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings

yolo11l.pt ready: 51.4 MB, sha256 9ebd0e09d59811db...
torch 2.8.0+cu128, CUDA available: True
GPU 0: Tesla T4, capability (7, 5), fp16 conv OK
GPU 1: Tesla T4, capability (7, 5), fp16 conv OK
GPUs visible: 2


In [7]:
# 5. the data
if not SKIP_DOWNLOAD:
    free_gb = shutil.disk_usage("/tmp").free / 1e9
    print(f"free space on /tmp: {free_gb:.0f} GB")
    if "[0-4]" in DATA_PATTERN and free_gb < 35:
        raise RuntimeError("Not enough scratch disk for the 50 files (~25 GB + margin).")
    try:  # the secret must be switched on for this notebook (Add-ons -> Secrets)
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        if "HF_TOKEN" not in os.environ:
            raise RuntimeError("HF_TOKEN secret not available. Add-ons -> Secrets -> switch it on.") from e
    sh(f"{VENV}/bin/hf auth whoami")                 # prints your username, never the token
    t = time.perf_counter()
    sh(f"{VENV}/bin/hf download HuggingFaceFV/finevideo --repo-type dataset "
       f"--include \"{DATA_PATTERN}\" --local-dir {DATA_ROOT}")
    print(f"download took {(time.perf_counter() - t) / 60:.1f} min")
files = sorted(glob.glob(f"{DATA_DIR}/*.parquet"))
print(len(files), "parquet files,", round(sum(os.path.getsize(f) for f in files) / 1e9, 1), "GB")
if "[0-4]" in DATA_PATTERN and len(files) != 50:
    print("WARNING: expected 50 files for the final run, found", len(files))
assert files, "no parquet files found"

free space on /tmp: 2692 GB
$ /tmp/venv/bin/hf auth whoami
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
✓ Logged in
  user: UrwaKhalid
$ /tmp/venv/bin/hf download HuggingFaceFV/finevideo --repo-type dataset --include "data/train-0000[0-1]-*" --local-dir /tmp/finevideo
✓ Downloaded
  path: /tmp/finevideo
download took 0.1 min
2 parquet files, 1.1 GB


## Optional: accuracy check (sampling vs every frame)
Set `RUN_ACCURACY_CHECK = True` in the config to run it. For each file in `ACC_FILES` it first runs the slow reference
(every frame, official Ultralytics predict, fp32), then your fast pipeline at each `ACC_K_VALUES`, and prints the brief's three scores.
Pick the smallest sampling rate that passes with margin (aim F1 >= 0.97, exact >= 0.90, within-1 >= 0.97).

In [8]:
if RUN_ACCURACY_CHECK:
    ACC = f"{OUT}/accuracy"
    os.makedirs(ACC, exist_ok=True)
    for fname in ACC_FILES:
        tag = fname.split("-of-")[0]
        folder = f"/tmp/acc_{tag}"
        os.makedirs(folder, exist_ok=True)
        link = f"{folder}/{fname}"
        if not os.path.exists(link):
            os.symlink(f"{DATA_DIR}/{fname}", link)
        ref = f"{ACC}/ref_{tag}.jsonl"
        sh(f"cd {REPO_DIR} && {PY} run.py --input {folder} --output {ref} --max-videos {ACC_VIDEOS} "
           f"--samples-per-sec 0 --detector official --fp32 --no-skip-nonref --profile{perf}")
        for k in ACC_K_VALUES:
            cand = f"{ACC}/k{k}_{tag}.jsonl"
            sh(f"cd {REPO_DIR} && {PY} run.py --input {folder} --output {cand} --max-videos {ACC_VIDEOS} "
               f"--samples-per-sec {k} --profile{perf} {EXTRA_ARGS}")
            print(f"\n===== {tag}: k = {k} frames per second =====")
            sh(f"cd {REPO_DIR} && {PY} tools/evaluate_results.py {ref} {cand} | grep -E 'micro|exact|within|PASS|FAIL'", check=False)
else:
    print("accuracy check skipped (RUN_ACCURACY_CHECK = False)")

$ cd /tmp/repo && /tmp/venv/bin/python run.py --input /tmp/acc_train-00000 --output /kaggle/working/accuracy/ref_train-00000.jsonl --max-videos 12 --samples-per-sec 0 --detector official --fp32 --no-skip-nonref --profile
done: 12 videos, 3692 lines (0 errors) in 1131.4s
  video time processed : 1.03 h  (3x real time)
  frames decoded       : 103949
  frames through model : 103949
  worker time, summed over workers (s) and as % of wall x workers:
    decode         149.8    6.6%
    preprocess       0.2    0.0%
    infer         2077.4   91.8%
$ cd /tmp/repo && /tmp/venv/bin/python run.py --input /tmp/acc_train-00000 --output /kaggle/working/accuracy/k2_train-00000.jsonl --max-videos 12 --samples-per-sec 2 --profile 
done: 12 videos, 3692 lines (0 errors) in 89.5s
  video time processed : 1.03 h  (41x real time)
  frames decoded       : 43116
  frames through model : 7377
  worker time, summed over workers (s) and as % of wall x workers:
    decode          50.9   28.4%
    preprocess  

In [9]:
# 6. the run
results = f"{OUT}/results.jsonl"
cmd = f"cd {REPO_DIR} && {PY} run.py --input {DATA_DIR} --output {results} --profile{extra}"
if MAX_VIDEOS:
    cmd += f" --max-videos {MAX_VIDEOS}"
t0 = time.perf_counter()
rc = sh(cmd, check=False)
wall = time.perf_counter() - t0
print(f"\nrun.py exit code {rc}; wall time {wall:.0f} s = {wall / 60:.1f} min")
if rc != 0:
    print("!!! run.py FAILED. Read the log above. Partial results are kept; re-run with --resume (EXTRA_ARGS = '--resume').")

$ cd /tmp/repo && /tmp/venv/bin/python run.py --input /tmp/finevideo/data --output /kaggle/working/results.jsonl --profile  --max-videos 6
done: 6 videos, 1576 lines (0 errors) in 65.0s
  video time processed : 0.44 h  (24x real time)
  frames decoded       : 20030
  frames through model : 6287
  worker time, summed over workers (s) and as % of wall x workers:
    decode          26.4   20.3%
    preprocess       2.6    2.0%
    infer           83.0   63.9%

run.py exit code 0; wall time 65 s = 1.1 min


In [10]:
# 7. check and package the results
videos, seconds, errors, bad_lines = set(), 0, {}, 0
with open(results) as f:
    for line in f:
        try:
            d = json.loads(line)
        except Exception:
            bad_lines += 1
            continue
        videos.add(d["video"])
        if "error" in d:
            errors[d["error"][:70]] = errors.get(d["error"][:70], 0) + 1
        else:
            seconds += 1
with open(results, "rb") as src, gzip.open(results + ".gz", "wb", compresslevel=6) as dst:
    shutil.copyfileobj(src, dst)
summary = dict(videos=len(videos), seconds=seconds, hours=round(seconds / 3600, 2), error_lines=sum(errors.values()),
               unparsable_lines=bad_lines, run_wall_seconds=round(wall), run_exit_code=rc, max_videos=MAX_VIDEOS,
               data_pattern=DATA_PATTERN, extra_args=extra.strip(), gpus=n_gpu,
               code_version=subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                                           capture_output=True, text=True).stdout.strip())
json.dump(summary, open(f"{OUT}/run_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))
print("error messages:", errors or "none")
print("files in", OUT, ":", {n: f"{os.path.getsize(os.path.join(OUT, n)) / 1e6:.1f} MB"
                              for n in sorted(os.listdir(OUT)) if os.path.isfile(os.path.join(OUT, n))})
if MAX_VIDEOS == 0 and "[0-4]" in DATA_PATTERN:
    print("\nFinal-run sanity: expect about 1,600 videos and ~450,000 seconds. Got", len(videos), "videos,", seconds, "seconds.")
else:
    print("\nNOTE: this was a partial / test run (MAX_VIDEOS or DATA_PATTERN is not the final setting). It is NOT the final results file.")

{
  "videos": 6,
  "seconds": 1576,
  "hours": 0.44,
  "error_lines": 0,
  "unparsable_lines": 0,
  "run_wall_seconds": 65,
  "run_exit_code": 0,
  "max_videos": 6,
  "data_pattern": "data/train-0000[0-1]-*",
  "extra_args": "",
  "gpus": 2,
  "code_version": "a481786"
}
error messages: none
files in /kaggle/working : {'__notebook__.ipynb': '0.1 MB', 'hardware_info.txt': '0.0 MB', 'results.jsonl': '0.1 MB', 'results.jsonl.gz': '0.0 MB', 'run_log.txt': '0.0 MB', 'run_summary.json': '0.0 MB'}

NOTE: this was a partial / test run (MAX_VIDEOS or DATA_PATTERN is not the final setting). It is NOT the final results file.
